# 04 – Evaluation & Interpretation

**Goal:** test the final model on data it has **never seen** and explain what the results mean
for the business / users.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable (notebooks/ is one level down)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ml import config

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

import json
import joblib
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, classification_report,
                             precision_recall_curve)
from sklearn.model_selection import train_test_split

from ml import train

In [ ]:
bundle = joblib.load(config.MODEL_PATH)
pipe = bundle["pipeline"]
report = json.loads(config.METRICS_PATH.read_text())
print("Model:", report["best_model"], report["best_params"])
pd.Series(report["test_metrics"])

Re-create the same test set (same `random_state`).

In [ ]:
df = train.load_dataset()
X, y, _ = train.prepare_xy(df)
stratify = y if config.TASK_TYPE == "classification" else None
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE, stratify=stratify)

## 1. Confusion matrix and ROC curve

In [ ]:
labels = [config.NEGATIVE_CLASS, config.POSITIVE_CLASS]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_estimator(pipe, X_test, y_test, display_labels=labels, cmap="Blues", ax=axes[0])
RocCurveDisplay.from_estimator(pipe, X_test, y_test, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "--", color="grey")
plt.tight_layout(); plt.show()
print(classification_report(y_test, pipe.predict(X_test), target_names=labels))

## 2. Choosing the decision threshold

By default a customer is predicted **Yes** when probability ≥ 0.5.
If missing a churner is costly, a lower threshold catches more of them (higher recall)
but creates more false alarms (lower precision).

In [ ]:
proba = pipe.predict_proba(X_test)[:, 1]
prec, rec, thr = precision_recall_curve(y_test, proba)
plt.plot(thr, prec[:-1], label="precision")
plt.plot(thr, rec[:-1], label="recall")
plt.axvline(0.5, ls="--", color="grey")
plt.xlabel("threshold"); plt.legend(); plt.title("Precision / recall vs threshold")
plt.show()

## 3. Error analysis – where is the model wrong?

In [ ]:
errors = X_test.assign(actual=y_test.values, prob_yes=proba.round(3), predicted=(proba >= 0.5).astype(int))
wrong = errors[errors["actual"] != errors["predicted"]]
print(f"{len(wrong)} wrong predictions out of {len(errors)}")
wrong.sort_values("prob_yes").head(10)

In [ ]:
# Error rate by contract type – is the model worse for some groups?
errors.assign(wrong=errors["actual"] != errors["predicted"]).groupby("contract_type")["wrong"].mean().round(3)

## 4. What drives the predictions?

In [ ]:
pd.Series(report["feature_importance"]).sort_values().plot.barh(figsize=(7, 6), color="#4472C4")
plt.title("Feature importance"); plt.show()

## 5. Conclusion ✍️

👉 TODO: answer in plain language (for a non-technical manager):

- How good is the model? (e.g. "it finds 77 % of customers who will leave")
- What are the main reasons customers leave?
- What should the company do with these predictions?
- Limitations: what data is missing, which groups the model is less accurate for.